In [2]:
"""
applied_mask_hidrica.py
=======================
Módulo de aplicação da máscara hídrica (gerada por `get_mask_hidrica.py`)
sobre os rasters recortados da área de estudo (produto do `cut_area.py`).

Fluxo principal:
    1. Pareamento automático entre arquivos de máscara hídrica (.tif) e
       rasters de cena (.tif) pelo identificador temporal GOES-16.
    2. Leitura da máscara hídrica: pixels de água (0) e sem dado (255)
       tornam-se NaN.
    3. Multiplicação elementar: pixels inválidos propagam NaN ao resultado.
    4. Exportação como GeoTIFF float32 com compressão LZW.

Convenção de máscara hídrica (gerada por get_mask_hidrica.py):
    0   → água / superfície não-vegetada  → NaN no resultado
    1   → vegetação / solo válido         → mantido no resultado
    255 → sem dado (nodata)               → NaN no resultado

Dependências:
    numpy, rasterio

Uso típico:
    batch_process(
        mask_files=mask_files,
        scene_files=scene_files,
        output_dir='FINAL',
    )
"""

import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import rasterio

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Padrão de chave temporal GOES-16: G16_s<CHAVE>_e
_PADRAO_CHAVE = re.compile(r"G16_s(\d+)_e")

# Valores da máscara hídrica que devem tornar-se NaN no resultado
# (água=0 e nodata=255, conforme convenção de get_mask_hidrica.py)
MASK_INVALID_VALUES = (0, 255)

# dtype de saída: float32 suficiente para dados de reflectância GOES-16
OUTPUT_DTYPE = np.float32


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _extract_key(filename: str) -> Optional[str]:
    """
    Extrai o identificador temporal do nome de um arquivo GOES-16.

    Parâmetros:
        filename (str): Nome do arquivo (sem caminho completo).

    Retorna:
        str | None: Chave temporal extraída, ou None se não encontrada.

    Exemplos:
        >>> _extract_key('ABI-L2-CMIPF_G16_s20201821600164_e20201821609472.tif')
        '20201821600164'
    """
    match = _PADRAO_CHAVE.search(filename)
    return match.group(1) if match else None


def _build_output_filename(scene_path: str) -> str:
    """
    Constrói o nome do arquivo GeoTIFF de saída a partir do nome da cena.

    Preserva o identificador temporal completo para evitar sobrescrita de
    arquivos de datas diferentes. Remove o prefixo 'clip_' se presente.

    Parâmetros:
        scene_path (str): Caminho completo ou nome do arquivo de cena.

    Retorna:
        str: Nome do arquivo de saída com extensão `.tif`.

    Exemplos:
        >>> _build_output_filename('clip_ABI-L2-CMIPF_G16_s20201821600164_e(...).tif')
        'ABI-L2-CMIPF_G16_s20201821600164_e(...).tif'
    """
    stem = Path(scene_path).stem

    # Remove prefixo 'clip_' herdado do módulo cut_area.py, se presente
    if stem.startswith("clip_"):
        stem = stem[len("clip_"):]

    return f"{stem}.tif"


def _apply_water_mask(
    scene_data: np.ndarray,
    mask_data: np.ndarray,
) -> np.ndarray:
    """
    Aplica a máscara hídrica sobre os dados de cena.

    Pixels com valor 0 (água) ou 255 (nodata) na máscara tornam-se NaN
    no resultado. Pixels com valor 1 (válido) são mantidos.

    Parâmetros:
        scene_data (np.ndarray): Array 2D de reflectância/radiância (float32).
        mask_data  (np.ndarray): Array 2D da máscara hídrica (uint8).

    Retorna:
        np.ndarray: Array float32 com pixels inválidos substituídos por NaN.
    """
    # Converte máscara para float32; valores inválidos → NaN
    mask_float = mask_data.astype(np.float32)
    for invalid in MASK_INVALID_VALUES:
        mask_float = np.where(mask_data == invalid, np.nan, mask_float)

    # Normaliza pixels válidos para 1.0 (máscara multiplicativa)
    mask_float = np.where(mask_data == 1, 1.0, mask_float)

    return (scene_data * mask_float).astype(OUTPUT_DTYPE)


# ---------------------------------------------------------------------------
# Funções públicas
# ---------------------------------------------------------------------------

def match_files(
    mask_files: List[str],
    scene_files: List[str],
) -> List[Tuple[str, str]]:
    """
    Emparelha arquivos de máscara hídrica e de cena pelo identificador temporal.

    Parâmetros:
        mask_files  (list[str]): Caminhos dos arquivos de máscara hídrica (.tif).
        scene_files (list[str]): Caminhos dos arquivos de cena (.tif).

    Retorna:
        list[tuple[str, str]]: Pares (mask_path, scene_path) com chave coincidente.
    """
    mask_index: Dict[str, str] = {
        key: path
        for path in mask_files
        if (key := _extract_key(os.path.basename(path)))
    }

    return [
        (mask_index[key], path)
        for path in scene_files
        if (key := _extract_key(os.path.basename(path))) and key in mask_index
    ]


def process_and_export(
    mask_path: str,
    scene_path: str,
    output_dir: str,
) -> None:
    """
    Aplica a máscara hídrica sobre uma cena e exporta o resultado como GeoTIFF.

    Parâmetros:
        mask_path  (str): Caminho da máscara hídrica (.tif).
        scene_path (str): Caminho do raster de cena (.tif).
        output_dir (str): Diretório de saída do GeoTIFF resultante.

    Levanta:
        FileNotFoundError: Se máscara ou cena não existirem.
    """
    for path in (mask_path, scene_path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Arquivo não encontrado: '{path}'")

    output_path = os.path.join(output_dir, _build_output_filename(scene_path))

    # Leitura da máscara hídrica (uint8: 0=água, 1=válido, 255=nodata)
    with rasterio.open(mask_path) as src:
        mask_data = src.read(1)

    # Leitura da cena — perfil usado como base para o GeoTIFF de saída
    with rasterio.open(scene_path) as src:
        scene_data = src.read(1).astype(OUTPUT_DTYPE)
        profile    = src.profile.copy()

    # Aplicação da máscara: água e nodata → NaN
    result = _apply_water_mask(scene_data, mask_data)

    # Atualiza perfil: dtype fixo, nodata declarado, otimizações de escrita
    profile.update({
        "driver":   "GTiff",
        "count":    1,
        "dtype":    "float32",
        "nodata":   np.nan,
        "compress": "lzw",
        "tiled":    True,
    })

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(result, 1)


def batch_process(
    mask_files: List[str],
    scene_files: List[str],
    output_dir: str,
    start_index: int = 0,
    verbose: bool = True,
) -> int:
    """
    Processa em lote os pares de máscara hídrica e cena correspondentes.

    Parâmetros:
        mask_files  (list[str]): Caminhos dos arquivos de máscara hídrica.
        scene_files (list[str]): Caminhos dos arquivos de cena.
        output_dir  (str):       Diretório de saída dos GeoTIFFs.
        start_index (int):       Índice inicial do processamento (padrão: 0).
        verbose     (bool):      Se True, exibe progresso por arquivo.

    Retorna:
        int: Número de pares processados com sucesso.
    """
    pairs = match_files(mask_files, scene_files)

    if not pairs:
        print("⚠️  Nenhum par correspondente encontrado.")
        return 0

    pairs_to_process = pairs[start_index:]
    total_pairs      = len(pairs)
    total_to_process = len(pairs_to_process)

    if verbose:
        print(f"Pares encontrados  : {total_pairs}")
        print(f"A processar        : {total_to_process} (a partir do índice {start_index})\n")

    success = 0
    for i, (mask_file, scene_file) in enumerate(pairs_to_process, start=start_index + 1):
        try:
            process_and_export(mask_file, scene_file, output_dir)
            success += 1
            if verbose:
                out_name = _build_output_filename(scene_file)
                print(f"  ✔ [{i}/{total_pairs}] {os.path.basename(scene_file)} → {out_name}")
        except Exception as e:
            print(f"  ✘ [{i}/{total_pairs}] {os.path.basename(scene_file)} — Erro: {e}")

    print(f"\nConcluído: {success}/{total_to_process} pares processados com sucesso.")
    return success


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    MASK_DIR   = Path("MASCARA_HIDRICA")
    SCENE_DIR  = Path("CUT")
    OUTPUT_DIR = Path("FINAL")

    # Valida diretórios de entrada antes de listar arquivos
    for directory in (MASK_DIR, SCENE_DIR):
        if not directory.exists():
            raise FileNotFoundError(f"Diretório não encontrado: '{directory}'")

    # Listagem ordenada para comportamento determinístico
    mask_files = sorted(
        str(f) for f in MASK_DIR.iterdir()
        if f.is_file() and f.suffix.lower() == ".tif"
    )
    scene_files = sorted(
        str(f) for f in SCENE_DIR.iterdir()
        if f.is_file() and f.suffix.lower() == ".tif"
    )

    if not mask_files or not scene_files:
        print("⚠️  Nenhum arquivo encontrado em um ou ambos os diretórios.")
    else:
        batch_process(
            mask_files=mask_files,
            scene_files=scene_files,
            output_dir=str(OUTPUT_DIR),
            start_index=0,
        )

Pares encontrados  : 60
A processar        : 60 (a partir do índice 0)

  ✔ [1/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501300166_e20201501309474_c20201501309548.tif → ABI-L2-CMIPF-M6C01_G16_s20201501300166_e20201501309474_c20201501309548.tif
  ✔ [2/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501310166_e20201501319474_c20201501319550.tif → ABI-L2-CMIPF-M6C01_G16_s20201501310166_e20201501319474_c20201501319550.tif
  ✔ [3/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501320166_e20201501329474_c20201501329547.tif → ABI-L2-CMIPF-M6C01_G16_s20201501320166_e20201501329474_c20201501329547.tif
  ✔ [4/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501330166_e20201501339474_c20201501339551.tif → ABI-L2-CMIPF-M6C01_G16_s20201501330166_e20201501339474_c20201501339551.tif
  ✔ [5/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501340166_e20201501349474_c20201501349555.tif → ABI-L2-CMIPF-M6C01_G16_s20201501340166_e20201501349474_c20201501349555.tif
  ✔ [6/60] clip_ABI-L2-CMIPF-M6C01_G16_s20201501350166_e20201501359474_c20201501359555.t